In [ ]:
import pandas as pd
import numpy as np
import geopandas as gpd
import plotly.express as px
import plotly.graph_objects as go

import nbformat
import json
from urllib.request import urlopen

import shapely.geometry
import shapely.ops

In [ ]:
with urlopen('https://raw.githubusercontent.com/plotly/datasets/master/geojson-counties-fips.json') as response:
    counties = json.load(response)
    
#drop 'StateFIPS': '09'
for i in range(0, 3214):
    if counties['features'][i]['properties']['STATE'] == '09':
        del counties['features'][i]


#add planning regions
json_file = (r"C:\Users\annas\OneDrive\blslabormap\CT_Planning_Regions.geojson")

with open(json_file) as f:
    ct_counties = json.load(f)

ct_counties.pop("name", None)
ct_counties.pop("crs", None)

for i in range(0, 9):
    ct_counties['features'][i]['properties']['NAME'] = ct_counties['features'][i]['properties'].pop("PlanningRegion")
    ct_counties['features'][i]['properties']['COUNTY'] = ct_counties['features'][i]['properties'].pop("PlanningRegionFIPS")
    ct_counties['features'][i]['properties']['STATE'] = ct_counties['features'][i]['properties'].pop("StateFIPS")
    ct_counties['features'][i]['properties']['GEO_ID'] = "0500000US" + ct_counties['features'][i]['properties'].pop("PlanningRegionFIPS_GEOID")
    ct_counties['features'][i]['properties'].pop("OBJECTID", None)
    ct_counties['features'][i]['properties'].pop("SHAPE__Area", None)
    ct_counties['features'][i]['properties'].pop("SHAPE__Length", None)
    ct_counties['features'][i]['properties']['LSAD'] = "County"
    ct_counties['features'][i]['properties']['CENSUSAREA'] = "None"


for i in range(0, 9):
    ct_counties['features'][i]['id'] = str(ct_counties['features'][i]['properties']['STATE'] + ct_counties['features'][i]['properties']['COUNTY'])

for i in range(0,9):
    counties['features'].append(ct_counties['features'][i])
#add oglala

oglala = { "type": "Feature", "properties": { "STATEFP": "46", "COUNTYFP": "102", "COUNTYNS": "01266992", "AFFGEOID": "0500000US46102", "GEOID": "46102", "NAME": "Oglala Lakota", "LSAD": "06", "ALAND": 5422404947, "AWATER": 7126155 }, "geometry": { "type": "Polygon", "coordinates": [ [ [ -103.001017, 43.605815999057796 ], [ -102.9025, 43.674887999057695 ], [ -102.809398, 43.687559999057683 ], [ -102.175231, 43.687755999057686 ], [ -102.139158, 43.700947999057654 ], [ -102.139051, 43.476675999058003 ], [ -102.110819, 43.389901999058168 ], [ -102.108892, 43.129257999058666 ], [ -102.082486319192, 42.99914054757825 ], [ -102.082548043072009, 42.999140639448953 ], [ -102.40864, 42.999625999058956 ], [ -102.792110939839006, 43.000036234829345 ], [ -103.000609526033003, 43.000259285833359 ], [ -103.000913, 43.476847999058016 ], [ -103.001017, 43.605815999057796 ] ] ] }, 'id': '46102'}
counties['features'].append(oglala)

In [ ]:
## unemployment file

In [ ]:
#unemployment = pd.read_csv(r"C:\Users\annas\OneDrive\blslabormap\data_frame_labor15_25.csv")

def add_zeros(string, num):
    to_add = num - len(string)
    return str((to_add * "0") + string)

unemployment['id'] = unemployment.apply(lambda x: (add_zeros(str(x['State FIPS Code'])[:-2], 2)) + (add_zeros(str(x['County FIPS Code'])[:-2], 3)),axis=1)
unemployment['Year'] = unemployment["Year"].astype('Int64')
unemployment = unemployment[unemployment['Year'].notna()]
unemployment['County Name'] = unemployment['County Name/State Abbreviation'].apply(lambda x: x[:-4])
unemployment['State Abbreviation'] = unemployment['County Name/State Abbreviation'].apply(lambda x: x[-2:])

In [ ]:
state_boundaries = gpd.read_file(r"C:\Users\annas\OneDrive\Github\plotly_bls_map\cb_2018_us_state_500k\cb_2018_us_state_500k.shp")
state_boundaries.to_file('myJson.geojson', driver='GeoJSON')
df = unemployment.groupby(['State FIPS Code', 'Year'])['Unemployment Rate (%)'].mean()
df = df.to_frame(name='Unemployment Rate Avg').reset_index()
df = df.rename(columns = {'State FIPS Code': 'STATEFP'})

In [ ]:
fig = px.choropleth_map(unemployment, 
                            geojson=counties, 
                            locations='id', 
                            color='Unemployment Rate (%)',
                           color_continuous_scale="sunsetdark",
                           range_color=(0, 12),
                           map_style="carto-positron",
                           zoom=3, center = {"lat": 37.0902, "lon": -95.7129},
                           animation_frame="Year", 
                           opacity=0.7,
                           custom_data = ['County Name', "Unemployment Rate (%)"]
                          )

for frame in fig.frames:
    for trace in frame.data:
        trace.hovertemplate = '<br>'.join(["%{customdata[0]}", "%{customdata[1]}%"])

fig.update_traces(hovertemplate = '<br>'.join(["%{customdata[0]}", "%{customdata[1]}%"]))
fig.update_layout(margin={"r":0,"t":0,"l":0,"b":0})
fig.update_traces(marker_line_width=0, marker_opacity=0.8)

#make map wider

state_boundaries = gpd.read_file(r"C:\Users\annas\OneDrive\Github\plotly_bls_map\cb_2018_us_state_500k\cb_2018_us_state_500k.shp")
state_boundaries.to_file('myJson.geojson', driver='GeoJSON')

fig.update_layout(
    coloraxis_showscale=False,
    map={
        "style":"carto-positron",
        "layers": [
            {
                "source": state_boundaries["geometry"].__geo_interface__,
                "type": "line",
                "color": "black"
            }
        ]
    },
)



In [ ]:
fig.show()